In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def analyze_lnn_memory_impact(
    neuron_counts=[64],    # ニューロン数
    input_dim=16,                     # 外部入力次元数
    
    # --- 1. Baseline: FP32 (Software / GPU Standard) ---
    fp32_bits=32,                     # 重みも状態もすべて32bit
    
    # --- 2. Competitor: INT8 (Standard Edge NPU) ---
    int8_bits=8,                      # 一般的な量子化モデル
    
    # --- 3. Proposed: LNN Accelerator (Ours) ---
    # CAM/LUTの設定
    prop_cam_in_bits=4,               # 提案: 入力は4bitに大胆に量子化（リストも可）
    prop_lut_out_bits=4,              # 提案: LUTの値は8bit（リストも可）
    prop_processor_sram_bits=8,            # 提案: その他プロセッサオーバーヘッドは8bit（リストも可）
    prop_processor_rram_bits=8,            # 提案: その他プロセッサオーバーヘッドは8bit（リストも可）
    # MVM(重み)の設定
    prop_weight_bits=4                # 提案: 重みは4bit（リストも可）
):
    """
    FP32, INT8, 提案手法の3つのメモリ容量を比較するシミュレーション
    各量子化パラメータはリストでも単一値でも可
    """
    
    results = []
    
    # 各パラメータがリストかどうか判定し、リスト化
    def ensure_list(x):
        return x if isinstance(x, (list, tuple, np.ndarray)) else [x]

    cam_bits_list = ensure_list(prop_cam_in_bits)
    lut_bits_list = ensure_list(prop_lut_out_bits)
    proc_sram_bits_list = ensure_list(prop_processor_sram_bits)
    proc_rram_bits_list = ensure_list(prop_processor_rram_bits)
    weight_bits_list = ensure_list(prop_weight_bits)

    for n_neuron in neuron_counts:
        for cam_bits in cam_bits_list:
            for lut_bits in lut_bits_list:
                for proc_sram_bits in proc_sram_bits_list:
                    for proc_rram_bits in proc_rram_bits_list:
                        for weight_bits in weight_bits_list:
                            # 共通パラメータ: 重みの総数
                            total_params_weight = (n_neuron + input_dim) * n_neuron
                            
                            # 1. Baseline: FP32 (Full Precision)
                            size_fp32_mvm = total_params_weight * fp32_bits
                            total_fp32_kbits = size_fp32_mvm / 1000
                            
                            # 2. Competitor: INT8 (Standard Quantization)
                            size_int8_mvm = total_params_weight * int8_bits * 2
                            size_int8_cam = (2**int8_bits) * int8_bits # CAM overhead
                            size_int8_lut = n_neuron * (2**int8_bits) * int8_bits # LUT overhead
                            size_int8_processor = 4 * n_neuron * int8_bits # その他プロセッサオーバーヘッド
                            total_int8_kbits = (size_int8_mvm + size_int8_cam + size_int8_lut + size_int8_processor) / 1000
                            
                            # 3. Proposed (Mixed Precision & Architecture)
                            size_prop_mvm = total_params_weight * weight_bits * 2
                            entries = 2 ** cam_bits
                            size_prop_cam = entries * cam_bits # Key
                            size_prop_lut = n_neuron * entries * lut_bits # Value
                            size_prop_processor = n_neuron * (3 * proc_rram_bits + proc_sram_bits) # その他プロセッサオーバーヘッド
                            total_prop_kbits = (size_prop_mvm + size_prop_cam + size_prop_lut + size_prop_processor) / 1000
                            
                            # 削減率の計算 (vs FP32)
                            reduction_vs_fp32 = (total_prop_kbits / total_fp32_kbits) * 100
                            reduction_vs_int8 = (total_prop_kbits / total_int8_kbits) * 100
                            
                            results.append({
                                "Neuron Count": n_neuron,
                                "CAM Size": size_prop_cam,
                                "LUT Size": size_prop_lut,
                                "Processor Size": size_prop_processor,
                                "MVM Size": size_prop_mvm,
                                "FP32 Size (kbits)": total_fp32_kbits,
                                "INT8 Size (kbits)": total_int8_kbits,
                                "Ours Size (kbits)": total_prop_kbits,
                                "Red. vs FP32 (%)": reduction_vs_fp32,
                                "Red. vs INT8 (%)": reduction_vs_int8
                            })
        return pd.DataFrame(results)

In [14]:
# -------------------------------------------------------
# 追加: CAM入力ビット数を7~2で変化させた場合の削減効果
# -------------------------------------------------------
bit_range = [8,7,6,5,4,3,2]
df_bitsweep = analyze_lnn_memory_impact(
    neuron_counts=[64],
    input_dim=19,
    prop_cam_in_bits=8,
    prop_weight_bits=8,
    prop_lut_out_bits=bit_range,
    prop_processor_sram_bits=8,
    prop_processor_rram_bits=8
)

print("\n=== 削減効果（CAM入力ビット数 7~2） ===")
print(df_bitsweep[["Neuron Count", "INT8 Size (kbits)", "Ours Size (kbits)", "Red. vs INT8 (%)", "CAM Size", "LUT Size", "Processor Size", "MVM Size"]].to_markdown(index=False, floatfmt=".2f"))


=== 削減効果（CAM入力ビット数 7~2） ===
|   Neuron Count |   INT8 Size (kbits) |   Ours Size (kbits) |   Red. vs INT8 (%) |   CAM Size |   LUT Size |   Processor Size |   MVM Size |
|---------------:|--------------------:|--------------------:|-------------------:|-----------:|-----------:|-----------------:|-----------:|
|          64.00 |              220.16 |              220.16 |             100.00 |    2048.00 |  131072.00 |          2048.00 |   84992.00 |
|          64.00 |              220.16 |              203.78 |              92.56 |    2048.00 |  114688.00 |          2048.00 |   84992.00 |
|          64.00 |              220.16 |              187.39 |              85.12 |    2048.00 |   98304.00 |          2048.00 |   84992.00 |
|          64.00 |              220.16 |              171.01 |              77.67 |    2048.00 |   81920.00 |          2048.00 |   84992.00 |
|          64.00 |              220.16 |              154.62 |              70.23 |    2048.00 |   65536.00 |          